[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session4/BMEN35_Ex4_02_What_changes_a_text_prediction.ipynb)

# What changes a text prediction?

**Run in Colab with the default CPU runtime.** Start at the top and run the cells in order. The first model load downloads about 18 MB of weights. We use a small pretrained BERT model from Hugging Face. You will see the exact text pieces and numbers it receives, then compare how the same model reacts to small changes in the input.

## What is SST-2?

**SST-2** is the two-class version of the **Stanford Sentiment Treebank**, a collection of English text from movie reviews annotated for sentiment. In the binary task, the two labels are **negative** (0) and **positive** (1). Neutral examples were left out of this binary version. Here a **checkpoint** means a saved version of a model's learned weights. The checkpoint we load was fine-tuned on a subset of SST-2, so it learned to give one of those two labels to a sentence.

We start with invented film reviews, then use invented comments about devices. The device comments let us check how the same model behaves on a different kind of text. Each sentence is classified independently. A score reflects how strongly this model favours a label for that sentence.

[Stanford Sentiment Treebank dataset description](https://huggingface.co/datasets/stanfordnlp/sst2) · [Checkpoint model card](https://huggingface.co/Facececersek/bedrock-sentiment-tiny).

In [ ]:
# Install the library in this notebook's Python environment.
%pip -q install transformers

## 1. Load the tokenizer and sentiment model

`pipeline` combines several steps. It tokenizes text, runs the model, converts its two class scores with softmax and returns a label with a score. We load the components once. The next section calls the tokenizer separately so we can inspect the input before we call the full pipeline.

The model's configuration maps a class ID such as `0` to a class name. A class ID refers to an **output** of the classifier. A token ID refers to a piece of **input** text in the tokenizer's vocabulary. They are two different lists of IDs.

In [ ]:
from transformers import pipeline                         # Load the helper that connects tokenization and inference.

model_name = 'Facececersek/bedrock-sentiment-tiny'           # Identify the saved sentiment checkpoint to download.
classifier = pipeline(                                    # Create one ready-to-use classifier.
    'text-classification',                                # Ask the pipeline to return a class for a sentence.
    model=model_name,                                     # Use the model trained for this specific task.
    device=-1,                                            # Run on CPU.
)

print('Model label names:', classifier.model.config.id2label)  # Inspect how the model names its output classes.

## 2. Inspect the actual tokens and their IDs

A **token** is a piece of input text from the tokenizer's vocabulary. The vocabulary is a table that gives each recognised piece an integer **token ID**. An ID is a lookup key for the embedding vector the model uses. A larger ID does not imply a more important token. A token's **position** says where it is in this example. A token keeps the same ID when it occurs at another position with the same token spelling.

Run the next cell and compare **The film was good.** with **The film was not good.** In this BERT tokenizer, `[CLS]` marks the start and gives the classifier a position for representing the sentence. `[SEP]` marks the end of this single sentence. These special tokens get IDs just like the pieces that come from the text. Because this tokenizer is uncased, uppercase letters can appear lowercased in the output.

Run the device example last. BERT can represent uncommon words with smaller pieces. A piece starting with `##` continues the current word. Look at how the phrase **pulse oximeter** is represented. Count the pieces and compare them with the number of words you would read in the sentence.

In [ ]:
original = 'The film was good.'                            # A short positive review.
with_negation = 'The film was not good.'                   # The same sentence with one word inserted.
device_example = 'The pulse oximeter was easy to use.'     # An example with a technical term.

for text in [original, with_negation, device_example]:     # Inspect one sentence at a time.
    encoded = classifier.tokenizer(text)                   # Apply the model's tokenizer and add special tokens.
    token_ids = encoded['input_ids']                       # Extract the ordered list of token IDs.
    tokens = classifier.tokenizer.convert_ids_to_tokens(token_ids)  # Look up each ID's vocabulary piece.

    print('\nText:', text)                                # Keep the original sentence visible above its tokens.
    for position, (piece, token_id) in enumerate(zip(tokens, token_ids)):  # Read corresponding pieces and IDs.
        print(f'  {position:2d}  {piece!r:14s}  ID {token_id}')  # Show position, readable token and integer ID.

### Look up neighbouring vocabulary entries

In the previous cell we started with a sentence and looked up its token IDs. Now go the other way. Look up the vocabulary entries with IDs 1000 through 1005 and compare them. The numbers identify entries in this particular tokenizer's vocabulary. Two neighbouring rows do not imply two synonymous tokens or similar meanings. The ID determines **which embedding row** to read. The learned numbers in that row supply the model with a representation of the piece.

In [ ]:
vocabulary_ids = list(range(1000, 1006))                   # Include IDs 1000, 1001, ..., 1005.
vocabulary_pieces = classifier.tokenizer.convert_ids_to_tokens(vocabulary_ids)  # Look up each piece by its ID.

for vocabulary_id, piece in zip(vocabulary_ids, vocabulary_pieces):  # Keep each ID next to its own piece.
    print(f'ID {vocabulary_id}: {piece!r}')                # Show punctuation and any piece markers explicitly.

**Pause.** Do any of the neighbouring entries make sense together as words? If two IDs are close, what does that actually tell you? It tells you that their entries are close in this vocabulary table. The numbers are not a measure of meaning. The same ID in a different tokenizer can name a different piece.

## 3. Compare two sentiment predictions

Read the two film sentences first and decide which label you expect for each. When the call `classifier(text)` returns, its list has one result for the one sentence we supplied. The dictionary inside contains the chosen label and the score for that **chosen label**. Some saved models expose generic names such as `LABEL_0`. For SST-2, the code maps `LABEL_0` to negative and `LABEL_1` to positive.

For a comparison we always want the score for **positive**. The two softmax scores add up to 1, so if the classifier reports negative with a score of 0.80, the positive score is `1 - 0.80 = 0.20`. If another sentence is classified as positive with a score of 0.80, its positive score is 0.80. Using the same output class for both lets you calculate a meaningful difference.

In the printout, `.3f` displays the score to three decimal places. Compare the label and the positive score. Adding **not** can lower the positive score even if the model still chooses positive.

In [ ]:
def get_prediction(text):                                 # Classify one sentence and return two comparable values.
    result = classifier(text)[0]                          # Get the first (and only) prediction in the returned list.
    label = result['label'].upper()                        # Read the predicted label and standardise the case.
    winning_score = result['score']                        # The score belongs to whichever label won.

    label_names = {'LABEL_0': 'NEGATIVE', 'LABEL_1': 'POSITIVE'}  # SST-2 uses these two class IDs.
    label = label_names.get(label, label)                  # Translate a generic name or keep the name we already have.

    positive_score = winning_score                        # If positive won, we already have the positive score.
    if label == 'NEGATIVE':                               # If negative won, its score is for the other class.
        positive_score = 1 - winning_score                # Of the two softmax scores, the other one is positive.

    return label, positive_score                          # Give the caller the label and the fixed-class score.


for text in [original, with_negation]:                    # Run the same function on each version of the sentence.
    label, positive_score = get_prediction(text)          # Save the two values returned by the function.
    print(f'{label:8s} | positive score {positive_score:.3f} | {text}')  # Keep the sentence beside its result.

## 4. Run three small experiments

Each row in `pairs` has five items. The first names the type of change. The second and third are sentences **A** and **B**. The fourth and fifth are the **reference labels**, which we chose before running the classifier. In the loop, those five items go into five variables in the same order.

- **Negation.** Add **not** to the first sentence. The intended sentiment goes from positive to negative.
- **Rewording.** Express the same positive comment in different words. Both reference labels are positive.
- **Technical term.** Replace **device** with **pulse oximeter**. The comment is still positive about how easy it was to use, so both reference labels are positive.

The change in positive score is **B minus A**. A negative result means the second sentence receives a lower positive score. A pair counts as correct when **both** predicted labels match their own reference labels. If the model calls both sentences negative in a pair whose reference labels are both positive, the labels are consistent with each other but neither is correct.

**After running.** Find the pair with the largest absolute change in positive score. Read both labels to see whether the score change was large enough to flip the model's choice.

In [ ]:
pairs = [                                                 # Each tuple describes one paired comparison.
    ('Negation',                                          # The expected label changes after inserting not.
     'The display was easy to read.',                      # A is the original sentence.
     'The display was not easy to read.',                  # B is the changed sentence.
     'POSITIVE', 'NEGATIVE'),                              # Reference labels for A and B, in that order.
    ('Rewording',                                         # This time both sentences express the same opinion.
     'The display was easy to read.',
     'I could read the display without difficulty.',
     'POSITIVE', 'POSITIVE'),                              # The intended label stays the same.
    ('Technical term',                                    # Only the device name changes.
     'The device was easy to use.',
     'The pulse oximeter was easy to use.',
     'POSITIVE', 'POSITIVE'),                              # The intended label stays the same again.
]

for experiment, text_a, text_b, expected_a, expected_b in pairs:  # Unpack one tuple in the order shown above.
    label_a, positive_a = get_prediction(text_a)          # Predict A and keep its positive score.
    label_b, positive_b = get_prediction(text_b)          # Predict B in exactly the same way.
    difference = positive_b - positive_a                  # Negative means the positive score went down.
    both_correct = label_a == expected_a and label_b == expected_b  # Both predictions must match.

    print(f'\n{experiment}')                             # Start a separate block for this comparison.
    print(f'A: {text_a} | expected {expected_a}, got {label_a} ({positive_a:.3f} positive)')  # Show A.
    print(f'B: {text_b} | expected {expected_b}, got {label_b} ({positive_b:.3f} positive)')  # Show B.
    print(f'Change in positive score (B - A): {difference:+.3f}')  # Include the sign of the change.
    print('Both labels correct:', both_correct)            # Show a single result for the pair.

### Read the score and the label together

Think through two hypothetical pairs before trying your own. If the positive score goes from 0.80 to 0.55, the model can still choose positive twice. It favours the label less strongly on the second sentence. A move from 0.51 to 0.49 can flip the label even though the score moves only 0.02. If the reference labels are both positive, the first pair gets both labels right and the second pair misses one.

When you read the output for the real examples, ask two different questions. Did the model move in the direction you expected? Did it get both labels right? A lower positive score after adding **not** is evidence of a move in the expected direction for our example. Both predicted labels could still be positive.

## 5. Try two new pairs

1. Add a pair with a negative sentence A. Reword sentence B so that its meaning stays negative. Write the two reference labels before you run the classifier.
2. Add a device-name substitution. Change **pulse oximeter** to another device name and keep the rest of sentence B the same. Print its tokens with the method in section 2. Then compare its label and score with sentence A.

Read the two scores for each pair. If the label stayed the same, say whether the model became more or less positive and by how much. If a label changed, compare each predicted label with its own reference label. These experiments are checks of specific behaviours. A systematic comparison would use more examples of each change. [Read about the CheckList approach](https://aclanthology.org/2020.acl-main.442/).